# Tiny Sparks — ACE-Step **1.5** song test (FREE, Kaggle GPU)

Same test as the v1 notebook (The Colors Song: chorus + 4 verses + chorus), on the newer **ACE-Step 1.5**
(2B turbo in float32 on the free T4 — XL needs a 24 GB GPU; MIT licence), with prompts tuned for a **more natural singing voice**. 2 styles x 3 takes = 6 versions.

**Before you run (Kaggle):** Settings -> **Accelerator: GPU T4 x2** · Settings -> **Internet: ON** · then **Run All**.
- Cell 1 (install) takes ~5-10 min the first time: it builds its own Python 3.11 environment with `uv`,
  because Kaggle's Python 3.13 is too new for ACE-Step 1.5. Red "dependency" warnings are normal.
- The first generation downloads the XL model (~9 GB) + its helpers, then each take is ~1-3 min.
- Listen at the bottom, download **`colors_song_xl_takes.zip`** from the Output panel.

**MODEL** in the settings cell: `acestep-v15-xl-turbo` (default: 8 steps, fast, "very high" quality) or
`acestep-v15-xl-sft` (50 steps + guidance: slower, sometimes cleaner vocals — try it second).

In [ ]:
# 1) Install ACE-Step 1.5 in its own Python 3.11 environment (uv)
!nvidia-smi --query-gpu=name,memory.total --format=csv
# uv via pip (the curl installer puts it in a folder Kaggle doesn't use)
!pip install -q uv
!uv --version
!git clone -q https://github.com/ACE-Step/ACE-Step-1.5.git /kaggle/working/ACE-Step-1.5 || echo "already cloned"
# downloads its own Python 3.11 + all packages (~5-10 min). The last lines should say "Installed N packages".
!cd /kaggle/working/ACE-Step-1.5 && uv sync --python 3.11 2>&1 | tail -n 8
!test -x /kaggle/working/ACE-Step-1.5/.venv/bin/python && echo "READY: ACE-Step 1.5 environment OK" || echo "NOT READY: the install above failed, paste its output"

In [ ]:
# 2) SETTINGS — edit freely, then run cells 2 -> 6
import json
SETTINGS = dict(
    # T4 (free Kaggle GPU): float16 overflows (NaN = silent takes) and XL in float32 needs ~16 GB > 15 GB,
    # so use the 2B models in float32: "acestep-v15-turbo" (fast, 8 steps) or "acestep-v15-sft" (50 steps).
    # XL ("acestep-v15-xl-turbo" / "acestep-v15-xl-sft") only on a 24 GB+ GPU (set force_fp32=False there).
    model="acestep-v15-turbo",
    force_fp32=True,                   # T4 fix: run in float32 instead of float16
    duration=75,                       # seconds
    seeds=[11, 42, 2026],
    steps=50,                          # xl-sft only (turbo always uses 8)
    guidance=7.0,                      # xl-sft only: keep moderate, high values sound metallic
    disable_cudnn=True,                # T4 fix from the v1 run ("unable to find an engine")
    lyrics="""[Chorus]
Colors, colors, all around!
Colors up and colors down!
Red and blue and yellow too,
I see colors, how about you?

[Verse 1]
Red, red, red! Red like an apple!
Orange, orange! Orange like a carrot!
Yellow, yellow! Yellow like the sun!
Green, green, green! Green like a frog!

[Chorus]
Colors, colors, all around!
Colors up and colors down!
Red and blue and yellow too,
I see colors, how about you?
""",
    styles={
        # the instruments you liked + a voice described as a REAL singer, recorded in a studio
        "nursery": "children's song, nursery rhyme, happy, bouncy, ukulele, xylophone, glockenspiel, hand claps, "
                   "light drums, natural warm female vocal, real singer, soft breathy voice, gentle vibrato, "
                   "clear diction, intimate studio recording, warm acoustic mix",
        "kidspop": "kids pop, sing-along, catchy, piano, acoustic guitar, hand claps, kids choir backing vocals, "
                   "natural expressive female lead vocal, real singer, warm and smiling tone, studio recording, "
                   "polished radio mix",
    },
    bpm={"nursery": 118, "kidspop": 124},
)
json.dump(SETTINGS, open("/kaggle/working/settings15.json", "w"), indent=1)
print("settings saved")

In [ ]:
%%writefile /kaggle/working/gen15.py
# Runs INSIDE the ACE-Step 1.5 uv environment (Python 3.11). Loads the XL model once, makes every take.
import json, os, shutil, sys, time
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["MPLBACKEND"] = "Agg"   # Kaggle sets the notebook-inline backend, which this venv does not have
S = json.load(open("/kaggle/working/settings15.json"))
ROOT = "/kaggle/working/ACE-Step-1.5"
OUT = "/kaggle/working/out15"
sys.path.insert(0, ROOT); os.chdir(ROOT); os.makedirs(OUT, exist_ok=True)
# Models (~9 GB XL + helpers) do NOT fit in /kaggle/working (~20 GB limit, the venv already uses part of it):
# put them on the biggest temporary disk instead (not saved with the notebook output, re-downloaded per session).
os.makedirs("/kaggle/tmp", exist_ok=True)
CK_BASE = max(["/kaggle/tmp", "/tmp"], key=lambda d: shutil.disk_usage(d).free)
CKPT = os.path.join(CK_BASE, "acestep-checkpoints")
os.makedirs(CKPT, exist_ok=True)
os.environ["ACESTEP_CHECKPOINTS_DIR"] = CKPT
os.environ["HF_HOME"] = os.path.join(CK_BASE, "hf")
os.environ["MODELSCOPE_CACHE"] = os.path.join(CK_BASE, "modelscope")
shutil.rmtree(os.path.join(ROOT, "checkpoints"), ignore_errors=True)   # half-downloaded files from the failed run
print(f"models -> {CKPT}  ({shutil.disk_usage(CK_BASE).free / 1e9:.0f} GB free there)", flush=True)
# T4 fix: ACE-Step picks float16 on pre-Ampere GPUs, which overflows -> NaN. Force float32 (one-line patch).
if S.get("force_fp32"):
    f = os.path.join(ROOT, "acestep/core/generation/handler/init_service_orchestrator.py")
    src = open(f).read()
    if "self.dtype = torch.float16" in src:
        open(f, "w").write(src.replace("self.dtype = torch.float16", "self.dtype = torch.float32  # patched: T4 fp16 overflow"))
        print("patched: float32 instead of float16", flush=True)
import torch
if S["disable_cudnn"]:
    torch.backends.cudnn.enabled = False
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported(), flush=True)
from acestep.handler import AceStepHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music

dit = AceStepHandler()
msg, ok = dit.initialize_service(project_root=ROOT, config_path=S["model"], device="cuda",
                                 offload_to_cpu=True,   # XL on a 16 GB T4 needs offload (README GPU table)
                                 quantization=None)     # no int8: keep full voice quality
print(msg, flush=True)
if not ok:
    sys.exit("model init failed (see above)")
turbo = "turbo" in S["model"]
tag = S["model"].replace("acestep-v15-", "")

for name, caption in S["styles"].items():
    for seed in S["seeds"]:
        out = f"{OUT}/colors-{tag}-{name}-seed{seed}.wav"
        p = GenerationParams(
            task_type="text2music", caption=caption, lyrics=S["lyrics"], instrumental=False,
            vocal_language="en", bpm=S["bpm"].get(name), duration=S["duration"],
            inference_steps=8 if turbo else S["steps"],
            guidance_scale=1.0 if turbo else S["guidance"],
            shift=3.0 if turbo else 1.0,                 # README: turbo needs shift=3.0
            seed=seed,
            thinking=False,                              # no 5Hz LM: saves VRAM, keeps OUR lyrics + caption
            use_cot_metas=False, use_cot_caption=False, use_cot_language=False,
        )
        c = GenerationConfig(batch_size=1, use_random_seed=False, seeds=[seed], audio_format="wav")
        tmp = "/kaggle/working/tmp15"
        shutil.rmtree(tmp, ignore_errors=True)
        t = time.time()
        r = generate_music(dit, None, p, c, save_dir=tmp)
        if r.success and r.audios:
            a = r.audios[0]
            if a.get("path") and os.path.exists(a["path"]):
                shutil.move(a["path"], out)
            else:
                import soundfile as sf
                sf.write(out, a["tensor"].numpy().T, a.get("sample_rate", 48000))
            print(f"{os.path.basename(out)}  OK  ({time.time() - t:.0f}s)", flush=True)
        else:
            print(f"{os.path.basename(out)}  FAILED: {r.error or r.status_message}", flush=True)

In [ ]:
# 4) Generate (the first run also downloads the XL model, ~9 GB, to the big temporary disk)
# free /kaggle/working first: half-downloaded models from a failed run can fill it (Kaggle limit ~20 GB)
!rm -rf /kaggle/working/ACE-Step-1.5/checkpoints /kaggle/working/tmp15
!df -h /kaggle/working /tmp | sed 1d
!cd /kaggle/working/ACE-Step-1.5 && MPLBACKEND=Agg .venv/bin/python /kaggle/working/gen15.py

In [ ]:
# 5) Listen
import glob, os
from IPython.display import Audio, display, Markdown
for p in sorted(glob.glob("/kaggle/working/out15/*.wav")):
    display(Markdown(f"**{os.path.basename(p)}**")); display(Audio(p))

In [ ]:
# 6) Download: colors_song_xl_takes.zip appears in the Output panel / /kaggle/working
import os, shutil
if not os.path.isdir("/kaggle/working/out15") or not os.listdir("/kaggle/working/out15"):
    raise SystemExit("No takes yet: cell 4 did not generate anything (look at its output).")
shutil.make_archive("/kaggle/working/colors_song_xl_takes", "zip", "/kaggle/working/out15")
print("-> /kaggle/working/colors_song_xl_takes.zip")